# All-in-One 2D Image -> 3D tren Google Colab

Notebook nay gom ca 2 pipeline:

- `triposr_360`: tao object 360 dep bang TripoSR, phu hop vat the don.
- `depth_relief`: tao depth map, point cloud va mesh relief bang Depth Anything V2 + Open3D, phu hop canh/phong canh/demo depth.

Chay tu tren xuong duoi. O cell cau hinh, chon pipeline ban muon dung.

In [ ]:
# Cell 1 - Kiem tra GPU
!nvidia-smi

In [ ]:
# Cell 2 - Chon pipeline va cau hinh chat luong
# Lua chon: 'triposr_360' hoac 'depth_relief'
PIPELINE = 'triposr_360'

# Cau hinh TripoSR.
# Preset dep hon: mesh day hon + bake texture de bot cam giac tuong go/nhua.
# Neu T4 thieu VRAM, giam MC_RESOLUTION ve 192 va CHUNK_SIZE ve 2048.
TRIPOSR_MC_RESOLUTION = 320
TRIPOSR_CHUNK_SIZE = 4096
TRIPOSR_FOREGROUND_RATIO = 0.8
TRIPOSR_BAKE_TEXTURE = True
TRIPOSR_TEXTURE_RESOLUTION = 2048

# Cau hinh Depth Anything + Open3D. Neu thieu VRAM/RAM, giam max_side va poisson_depth.
DEPTH_MODEL_ID = 'depth-anything/Depth-Anything-V2-Small-hf'
MAX_IMAGE_SIDE = 768
POISSON_DEPTH = 8
TARGET_TRIANGLES = 60000
VOXEL_SIZE = 0.005

assert PIPELINE in ['triposr_360', 'depth_relief']
print('Pipeline:', PIPELINE)
print('TripoSR bake texture:', TRIPOSR_BAKE_TEXTURE)


In [ ]:
# Cell 3 - Cai dat thu vien theo pipeline da chon
!pip -q install --upgrade pip
!pip -q install "setuptools<82" wheel "jedi>=0.16"
!pip -q install torch torchvision --index-url https://download.pytorch.org/whl/cu124

if PIPELINE == 'triposr_360':
    !rm -rf /content/TripoSR
    !git clone https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR
    %cd /content/TripoSR
    !pip -q install -r requirements.txt
    !pip -q install "numpy==1.26.4" "cupy-cuda12x==13.6.0" onnxruntime trimesh
else:
    !pip -q install transformers accelerate timm opencv-python pillow matplotlib numpy open3d trimesh fastapi uvicorn pyngrok python-multipart nest_asyncio

In [ ]:
# Cell 4 - Import chung va tao thu muc input/output
import json
import os
import re
import uuid
from base64 import b64encode
from pathlib import Path

import numpy as np
import torch
import trimesh
from google.colab import files
from IPython.display import HTML, display
from PIL import Image

BASE_DIR = Path('/content/all_in_one_2d_to_3d')
INPUT_DIR = BASE_DIR / 'inputs'
OUTPUT_DIR = BASE_DIR / 'outputs'
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Output dir:', OUTPUT_DIR)

In [ ]:
# Cell 5 - Upload anh JPG/PNG
upload_cwd = INPUT_DIR
upload_cwd.mkdir(parents=True, exist_ok=True)
os.chdir(str(upload_cwd))
print('Upload dir:', upload_cwd)

uploaded = files.upload()
assert uploaded, 'Ban can upload 1 anh JPG/PNG.'

first_name = next(iter(uploaded.keys()))
input_path = INPUT_DIR / first_name
with open(input_path, 'wb') as f:
    f.write(uploaded[first_name])

print('Input:', input_path)


In [ ]:
# Cell 6A - Chay TripoSR neu PIPELINE = 'triposr_360'
if PIPELINE == 'triposr_360':
    import shutil
    import subprocess

    triposr_dir = Path('/content/TripoSR')
    run_py = triposr_dir / 'run.py'
    if not run_py.exists():
        raise FileNotFoundError('Khong tim thay /content/TripoSR/run.py. Hay chay lai Cell 3 de clone/cai TripoSR truoc khi chay Cell 6A.')

    tsr_output_dir = OUTPUT_DIR / 'triposr'
    if tsr_output_dir.exists():
        shutil.rmtree(tsr_output_dir)
    tsr_output_dir.mkdir(parents=True, exist_ok=True)

    model_save_format = 'obj' if TRIPOSR_BAKE_TEXTURE else 'glb'
    cmd = [
        'python',
        str(run_py),
        str(input_path),
        '--output-dir',
        str(tsr_output_dir),
        '--model-save-format',
        model_save_format,
        '--mc-resolution',
        str(TRIPOSR_MC_RESOLUTION),
        '--chunk-size',
        str(TRIPOSR_CHUNK_SIZE),
        '--foreground-ratio',
        str(TRIPOSR_FOREGROUND_RATIO),
    ]
    if TRIPOSR_BAKE_TEXTURE:
        cmd.extend(['--bake-texture', '--texture-resolution', str(TRIPOSR_TEXTURE_RESOLUTION)])

    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=str(triposr_dir), text=True)
    if result.returncode != 0:
        raise RuntimeError('TripoSR inference bi loi. Hay xem log phia tren; neu /content/TripoSR bi mat, chay lai Cell 3.')

    raw_mesh_path = tsr_output_dir / '0' / f'mesh.{model_save_format}'
    assert raw_mesh_path.exists(), f'TripoSR khong tao duoc {raw_mesh_path.name}. Hay xem log o cell nay.'

    glb_path = OUTPUT_DIR / 'mesh.glb'
    obj_path = OUTPUT_DIR / 'mesh.obj'
    ply_path = OUTPUT_DIR / 'mesh.ply'

    if TRIPOSR_BAKE_TEXTURE:
        texture_path = tsr_output_dir / '0' / 'texture.png'
        scene = trimesh.load(str(raw_mesh_path), force='scene')
        scene.export(str(glb_path))
        shutil.copy2(raw_mesh_path, obj_path)
        if texture_path.exists():
            shutil.copy2(texture_path, OUTPUT_DIR / 'texture.png')
        mesh_for_ply = trimesh.load(str(raw_mesh_path), force='mesh')
        mesh_for_ply.export(str(ply_path))
    else:
        shutil.copy2(raw_mesh_path, glb_path)
        mesh = trimesh.load(str(glb_path), force='mesh')
        mesh.export(str(obj_path))
        mesh.export(str(ply_path))

    artifacts = {
        'pipeline': 'triposr_360',
        'glb': str(glb_path),
        'obj': str(obj_path),
        'ply': str(ply_path),
    }
    if TRIPOSR_BAKE_TEXTURE and (OUTPUT_DIR / 'texture.png').exists():
        artifacts['texture_png'] = str(OUTPUT_DIR / 'texture.png')
    print(json.dumps(artifacts, indent=2))
else:
    print('Bo qua cell TripoSR vi PIPELINE =', PIPELINE)


In [ ]:
# Cell 6B - Chay Depth Anything + Open3D neu PIPELINE = 'depth_relief'
if PIPELINE == 'depth_relief':
    import cv2
    import matplotlib.pyplot as plt
    import open3d as o3d
    from transformers import AutoImageProcessor, AutoModelForDepthEstimation

    processor = AutoImageProcessor.from_pretrained(DEPTH_MODEL_ID)
    depth_model = AutoModelForDepthEstimation.from_pretrained(DEPTH_MODEL_ID).to(DEVICE)
    if DEVICE == 'cuda':
        depth_model = depth_model.half()
    depth_model.eval()

    def safe_stem(path_or_name: str) -> str:
        stem = Path(path_or_name).stem
        stem = re.sub(r'[^A-Za-z0-9_.-]+', '_', stem).strip('._')
        return stem or 'image'

    def open_and_resize_image(image_path: str) -> Image.Image:
        image_pil = Image.open(image_path).convert('RGB')
        w, h = image_pil.size
        scale = min(MAX_IMAGE_SIDE / max(w, h), 1.0)
        if scale < 1.0:
            image_pil = image_pil.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)
        return image_pil

    def estimate_depth(image_pil: Image.Image) -> np.ndarray:
        inputs = processor(images=image_pil, return_tensors='pt')
        inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
        with torch.no_grad():
            if DEVICE == 'cuda':
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    outputs = depth_model(**inputs)
            else:
                outputs = depth_model(**inputs)
        pred_depth = outputs.predicted_depth
        pred_depth = torch.nn.functional.interpolate(pred_depth.unsqueeze(1), size=image_pil.size[::-1], mode='bicubic', align_corners=False).squeeze()
        depth = pred_depth.detach().cpu().numpy()
        depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)
        return depth.astype(np.float32)

    def depth_to_point_cloud(image_rgb: np.ndarray, depth_norm: np.ndarray) -> o3d.geometry.PointCloud:
        h, w = depth_norm.shape
        depth_u16 = np.clip((1.0 - depth_norm) * 1200.0, 1, 65535).astype(np.uint16)
        color_o3d = o3d.geometry.Image(image_rgb)
        depth_o3d = o3d.geometry.Image(depth_u16)
        fx = fy = max(w, h) * 1.2
        intrinsic = o3d.camera.PinholeCameraIntrinsic(w, h, fx, fy, w / 2.0, h / 2.0)
        rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(color_o3d, depth_o3d, depth_scale=1000.0, depth_trunc=5.0, convert_rgb_to_intensity=False)
        pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsic)
        pcd.transform([[1, 0, 0, 0], [0, -1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]])
        pcd = pcd.voxel_down_sample(voxel_size=VOXEL_SIZE)
        pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=1.5)
        return pcd

    def colorize_mesh_from_point_cloud(mesh: o3d.geometry.TriangleMesh, pcd: o3d.geometry.PointCloud) -> o3d.geometry.TriangleMesh:
        if not pcd.has_colors() or len(mesh.vertices) == 0:
            return mesh
        kdtree = o3d.geometry.KDTreeFlann(pcd)
        pcd_colors = np.asarray(pcd.colors)
        vertex_colors = []
        for vertex in np.asarray(mesh.vertices):
            _, idx, _ = kdtree.search_knn_vector_3d(vertex, 1)
            vertex_colors.append(pcd_colors[idx[0]] if idx else [0.8, 0.8, 0.8])
        mesh.vertex_colors = o3d.utility.Vector3dVector(np.asarray(vertex_colors))
        return mesh

    def point_cloud_to_mesh(pcd: o3d.geometry.PointCloud) -> o3d.geometry.TriangleMesh:
        if len(pcd.points) < 100:
            raise ValueError('Point cloud qua it diem. Hay thu anh ro hon hoac giam VOXEL_SIZE.')
        pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.03, max_nn=30))
        pcd.orient_normals_consistent_tangent_plane(10)
        mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=POISSON_DEPTH)
        densities = np.asarray(densities)
        mesh.remove_vertices_by_mask(densities < np.quantile(densities, 0.05))
        mesh = mesh.crop(pcd.get_axis_aligned_bounding_box())
        if len(mesh.triangles) > TARGET_TRIANGLES:
            mesh = mesh.simplify_quadric_decimation(target_number_of_triangles=TARGET_TRIANGLES)
        mesh.remove_degenerate_triangles()
        mesh.remove_duplicated_triangles()
        mesh.remove_duplicated_vertices()
        mesh.remove_non_manifold_edges()
        mesh.compute_vertex_normals()
        return colorize_mesh_from_point_cloud(mesh, pcd)

    image_pil = open_and_resize_image(str(input_path))
    image_rgb = np.array(image_pil)
    depth_norm = estimate_depth(image_pil)
    pcd = depth_to_point_cloud(image_rgb, depth_norm)
    mesh = point_cloud_to_mesh(pcd)

    depth_png = OUTPUT_DIR / 'depth.png'
    ply_path = OUTPUT_DIR / 'point_cloud.ply'
    obj_path = OUTPUT_DIR / 'mesh.obj'
    glb_path = OUTPUT_DIR / 'mesh.glb'
    cv2.imwrite(str(depth_png), (depth_norm * 255).astype(np.uint8))
    o3d.io.write_point_cloud(str(ply_path), pcd)
    o3d.io.write_triangle_mesh(str(obj_path), mesh, write_triangle_uvs=False)

    colors = None
    if mesh.has_vertex_colors():
        colors = (np.asarray(mesh.vertex_colors) * 255).clip(0, 255).astype(np.uint8)
    tm = trimesh.Trimesh(vertices=np.asarray(mesh.vertices), faces=np.asarray(mesh.triangles), vertex_normals=np.asarray(mesh.vertex_normals), vertex_colors=colors, process=False)
    tm.export(str(glb_path))

    artifacts = {
        'pipeline': 'depth_relief',
        'depth_png': str(depth_png),
        'ply': str(ply_path),
        'obj': str(obj_path),
        'glb': str(glb_path),
    }
    print(json.dumps(artifacts, indent=2))
    print(f'Point cloud: {len(pcd.points)} points')
    print(f'Mesh: {len(mesh.vertices)} vertices, {len(mesh.triangles)} triangles')

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.title('Anh goc')
    plt.imshow(image_rgb)
    plt.axis('off')
    plt.subplot(1, 2, 2)
    plt.title('Depth map')
    plt.imshow(depth_norm, cmap='inferno')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Bo qua cell depth vi PIPELINE =', PIPELINE)

In [ ]:
# Cell 7 - Preview GLB trong Colab
glb_for_preview = artifacts['glb']
with open(glb_for_preview, 'rb') as f:
    glb_b64 = b64encode(f.read()).decode('utf-8')

display(HTML(f'''
<script type="module" src="https://unpkg.com/@google/model-viewer/dist/model-viewer.min.js"></script>
<model-viewer src="data:model/gltf-binary;base64,{glb_b64}" camera-controls auto-rotate shadow-intensity="1" environment-image="neutral" style="width:100%;height:560px;background:#eef2f7;border:1px solid #d8dee8;border-radius:8px;"></model-viewer>
'''))

In [ ]:
# Cell 8 - Tai file ket qua ve may
for key, path in artifacts.items():
    if key == 'pipeline':
        continue
    print('Download:', key, path)
    files.download(path)

## Goi y nhanh

- Muon 360 dep: de `PIPELINE = 'triposr_360'`.
- Muon depth/point cloud/phong canh: doi thanh `PIPELINE = 'depth_relief'`.
- T4 het VRAM voi TripoSR: dat `TRIPOSR_MC_RESOLUTION = 192`, `TRIPOSR_CHUNK_SIZE = 2048`.
- T4 het RAM voi depth: dat `MAX_IMAGE_SIDE = 512`, `POISSON_DEPTH = 7`, `TARGET_TRIANGLES = 30000`.